# Folding story with more pixels

A follow-up to [`story.ipynb`](story.ipynb): same idea (one printed square that shows a new picture at each stage of folding a crane), aimed at its biggest weakness. The story here: **home planet (flat) → rocket (bird base) → another planet (crane).**

## The problem: the crane gets very few pixels

DeepFloyd draws in two stages: a 64 × 64 image that decides the composition, then a 256 × 256 upscale that adds detail. Everything is drawn on the whole sheet, and folding decides what share of the sheet each stage shows. That share is fixed by geometry, so printing bigger doesn't help:

| | Stage 1 (64 × 64) | 128 × 128 | Stage 2 (256 × 256) |
|---|---|---|---|
| Bird base paper pixels | 848 | 3,392 | 13,572 |
| Crane paper pixels | **489** | 1,925 | 7,681 |

In `story.ipynb`, the crane's picture is composed in those 489 pixels (its body is about 16 × 22), and stage 2 can only sharpen what's there. That's why busy prompts came out unreadable on the crane.

## Two fixes, tested side by side

1. **Stage 2 redraws the stage pictures.** Stage 2 is told how far it may depart from stage 1 by a *noise level*: IF's default, 250, means "mostly keep it". Here the flat view keeps 250, and the folded views get a higher level, so stage 2 composes the rocket and the planet at 256 pixels instead of only sharpening them.
2. **Stage 1 at 128 × 128.** The model is convolutional, so it runs at 128 and every view gets four times the pixels. But it was trained at 64, so it may repeat or fragment things. Only a run will tell.

The experiments cell runs the baseline, each fix alone, and both together, with the same seed.

## Why only the crane

The plan was to also try Yakko (a traditional figure that shows more paper than the crane, in more pieces). But a story needs the in-between stages, and Flat-Folder's Yakko has none we can use. A search over every subset of its creases found that, keeping each crease's mountain or valley, only the full model and the flat sheet (with its corner tips folded in) fold flat. Its real folding sequence must pass through stages where some creases are folded the other way and later reversed. To add Yakko, we'd need its folding sequence (fold one and note the steps), then build each stage's crease pattern from that, as `make_stages.py` does for the crane.

## Setup

Clones [rath-hy/doodles](https://github.com/rath-hy/doodles/tree/main/2026-10-31-fold-map) on Colab, or uses the local files if you run this from that folder.

In [ ]:
import importlib, os, subprocess, sys

FOLDER = "2026-10-31-fold-map"
if not os.path.exists("story.py"):
    if not os.path.exists("doodles"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/rath-hy/doodles.git"], check=True)
    os.chdir(os.path.join("doodles", FOLDER))
if "google.colab" in sys.modules:
    # On Colab, rerunning this cell gets the latest code and reloads anything already imported.
    subprocess.run(["git", "pull", "-q"], check=True)
    for name in ["fold_map", "directions", "decal", "illusion", "story"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
sys.path.insert(0, os.getcwd())

import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from illusion import generate, printable
from story import least_shared_sides, load_stages, shared_paper, stage_views, story_images

stages, models = load_stages()                 # bird base, crane
SIDES = least_shared_sides(models)
names = [name for name, _, _, _ in stages]
for name, side, share in zip(names, SIDES, shared_paper(models, SIDES)):
    print(f"{name} ({side}): {share:.0%} of its visible paper also shows at the other stage")

## Install and log in (needs a GPU)

1. Use a **GPU runtime** (Runtime → Change runtime type; a T4 works).
2. On Hugging Face, accept the license for [DeepFloyd/IF-I-M-v1.0](https://huggingface.co/DeepFloyd/IF-I-M-v1.0) and [DeepFloyd/IF-II-M-v1.0](https://huggingface.co/DeepFloyd/IF-II-M-v1.0), and have an [access token](https://huggingface.co/settings/tokens) ready.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece
from huggingface_hub import notebook_login
notebook_login()

## The story

Choices made to raise the odds:

- **Home planet as an aerial view, not a globe.** The flat sheet is where the other two pictures hide, so it should be busy everywhere. A globe on black space would leave large plain areas where any hidden shape stands out (and black prints muddy). An aerial view of forests, mountains and rivers is texture from edge to edge, and still says "home planet" in blue, green, brown and gray.
- **The flat palette includes the later pictures' colors.** Orange autumn trees give the red rocket somewhere to hide; purple-gray mountain shadows do the same for the planet.
- **A red and white rocket, pointing up, for the bird base.** The bird base is a tall, narrow diamond, which a rocket fills naturally. A red and white rocket is the most iconic version, so the model draws it most reliably. A brown ship would hide better but read worse; reading matters more here.
- **A ringed planet for the crane.** The crane held this way is wider than it is tall, and a ring stretches across the wings. It's purple rather than gray, because the backdrop is light gray and gray prints dull.
- **Plain light backgrounds and short prompts** for the folded views, since they're drawn on a plain backdrop and have few pixels.

In [ ]:
STYLE = "a watercolor painting of "
FLAT_PROMPT = STYLE + "an aerial view of green forests, orange autumn trees, gray mountains and winding blue rivers"
STAGE_PROMPTS = [
    STYLE + "a red and white rocket ship flying upward, on a plain light background",   # bird base
    STYLE + "a purple planet with rings, on a plain light background",                  # crane
]
# Alternatives to try if a picture doesn't read:
#   rocket: "a silver rocket with orange flames", "a red rocket ship"
#   planet: "Saturn", "an orange planet with rings", "a cratered gray moon" (your original idea)
#   style:  "a colorful children's book illustration of " (bolder shapes, survives printing better)
assert len(STAGE_PROMPTS) == len(stages), f"one prompt per stage: {names}"

## Encode the prompts

The text encoder is loaded onto the GPU, used once, and deleted to save memory. **Rerun this cell whenever you change the prompts.** The first run downloads about 11 GB.

In [ ]:
import gc, torch
from transformers import T5EncoderModel
from diffusers import IFPipeline, IFSuperResolutionPipeline

skip = dict(safety_checker=None, feature_extractor=None, watermarker=None)

text_encoder = T5EncoderModel.from_pretrained("DeepFloyd/IF-I-M-v1.0", subfolder="text_encoder", variant="fp16",
                                              torch_dtype=torch.float16, device_map="cuda")
encoder = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=text_encoder, unet=None, **skip)
prompt_embeds, negative_embeds = encoder.encode_prompt([FLAT_PROMPT] + STAGE_PROMPTS)

del encoder, text_encoder
gc.collect(); torch.cuda.empty_cache()
print("embeddings:", tuple(prompt_embeds.shape))

## Load the image models

Once per session.

In [ ]:
if "stage_1" not in globals():
    stage_1 = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=None, variant="fp16",
                                         torch_dtype=torch.float16, **skip).to("cuda")
    stage_2 = IFSuperResolutionPipeline.from_pretrained("DeepFloyd/IF-II-M-v1.0", text_encoder=None, variant="fp16",
                                                        torch_dtype=torch.float16, **skip).to("cuda")

## Experiments

Each experiment sets:

- `res1`: the stage-1 resolution, 64 (what IF was trained at) or 128.
- `fold_level`: stage 2's noise level for the folded views (the flat view stays at 250). Higher means stage 2 redraws more. Around 1000, it ignores stage 1 there entirely.

Everything else matches `story.ipynb`. Stage 1 at 128 takes about four times as long as at 64. On a T4, expect a few minutes for all four.

In [ ]:
EXPERIMENTS = {
    "A. baseline":         dict(res1=64,  fold_level=250),
    "B. stage 2 redraws":  dict(res1=64,  fold_level=700),
    "C. stage 1 at 128":   dict(res1=128, fold_level=250),
    "D. both":             dict(res1=128, fold_level=700),
}
FOLD_WEIGHT = 0.5
GUIDANCE = (7.0, 4.0)
STEPS = (50, 40)
SEED = 0

views = {}                                     # FoldViews per resolution, made once
def views_at(res):
    if res not in views:
        views[res] = stage_views(stages, models, SIDES, res)
    return views[res]

results = {}
for label, ex in EXPERIMENTS.items():
    start = time.time()
    results[label] = generate(stage_1, stage_2, views_at(ex["res1"]), views_at(256), (negative_embeds, prompt_embeds),
                              guidance=GUIDANCE, steps=STEPS, fold_weight=FOLD_WEIGHT, seed=SEED,
                              noise_level=[250] + [ex["fold_level"]] * len(stages))
    print(f"{label}: {time.time() - start:.0f} s")

## Compare

One row per experiment: the printed sheet, then each stage folded from it with the decal simulator (independent of the maps used to make the sheet). The second figure shows the stage-1 sheets, where the composition is decided.

What to look for:

- **Does the planet read on the crane, and the rocket on the bird base?** That's what the fixes are for.
- **Does the flat sheet still look like one landscape,** or do rocket and planet fragments show through? Redrawing at stage 2 may cost some hiding, since stage 2 now pushes harder on those pixels.
- **At 128, is the flat view still one coherent scene,** or repeated, tiled patches? That's the risk of running stage 1 above its training size.

In [ ]:
cols = ["flat (home planet)"] + [f"{name} ({side})" for name, side in zip(names, SIDES)]
fig, axes = plt.subplots(len(results), len(cols), figsize=(4 * len(cols), 4 * len(results)), squeeze=False)
for i, (row, (label, (_, final))) in enumerate(zip(axes, results.items())):
    for ax, img, col in zip(row, story_images(final, stages, models, SIDES), cols):
        ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(col if i == 0 else "")
    row[0].set_ylabel(label, fontsize=13)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4.3))
for ax, (label, (first, _)) in zip(axes, results.items()):
    ax.imshow(first, interpolation="nearest")
    ax.set_title(f"{label}\nstage 1, {first.shape[0]} × {first.shape[0]}")
    ax.axis("off")
plt.show()

## Try more seeds

Seeds matter as much as settings, so once one experiment looks promising, run it on a few seeds before judging it.

In [ ]:
PICK = "D. both"
SEEDS = [1, 2, 3]

ex = EXPERIMENTS[PICK]
for seed in SEEDS:
    _, final = generate(stage_1, stage_2, views_at(ex["res1"]), views_at(256), (negative_embeds, prompt_embeds),
                        guidance=GUIDANCE, steps=STEPS, fold_weight=FOLD_WEIGHT, seed=seed,
                        noise_level=[250] + [ex["fold_level"]] * len(stages))
    results[f"{PICK}, seed {seed}"] = (None, final)
    fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 4.3))
    for ax, img, col in zip(axes, story_images(final, stages, models, SIDES), cols):
        ax.imshow(img); ax.set_title(f"seed {seed}: {col}"); ax.axis("off")
    plt.show()

## Print the best one

Saves the sheet, and an A4 PDF with two copies (148.5 mm squares), so you get two tries. Print at **actual size** and fold with the **printed side out**. The bird base is held with its wing flaps up.

In [ ]:
BEST = "D. both"                              # any key of results, e.g. "D. both, seed 2"

name = "planets_" + "_".join(BEST.replace(".", "").replace(",", "").split())     # e.g. planets_D_both_seed_2
img = printable(results[BEST][1], 1754)       # 148.5 mm at 300 dpi
img.save(name + ".png", dpi=(300, 300))
page = Image.new("RGB", (2480, 3508), "white")  # A4 at 300 dpi
page.paste(img, (0, 0))
page.paste(img, (0, 3508 - 1754))
page.save(name + "_a4.pdf", resolution=300)
print("saved", name + ".png", "and", name + "_a4.pdf")
try:
    from google.colab import files
    files.download(name + "_a4.pdf")
except ImportError:
    pass

## Notes

- **Stage 2 has its own ceiling.** IF's stage 2 is a super-resolution model, trained to sharpen rather than compose. At very high noise levels it composes from the prompt alone, which it does less well than stage 1. If B and D look mushy at 700, try 500.
- **The bigger fix, for later: mixed scales.** Generate the whole sheet at high resolution, so the folded views get their pixels one to one, and show the flat view a downscaled copy at 64. The catch is the noise: averaging blocks of pixels keeps the picture but shrinks the noise, which the model isn't trained for. Topping it up with fresh noise fixes the flat view's input, but then its prediction only covers the coarse image, and the fine detail in hidden areas needs a prediction from somewhere else. *Factorized Diffusion* (Geng, Park, Owens, 2024) deals with a similar split between coarse and fine. Getting this right would be a real research contribution.
- **A bigger stage 1 model.** `IF-I-L-v1.0` (900M parameters) instead of `IF-I-M-v1.0` (400M) composes better and may fit a T4. Swap the name in the load cell.

## References

- D. Geng, I. Park, A. Owens. [Visual Anagrams: Generating Multi-View Optical Illusions with Diffusion Models](https://dangeng.github.io/visual_anagrams/). *CVPR*, 2024.
- D. Geng, I. Park, A. Owens. [Factorized Diffusion: Perceptual Illusions by Noise Decomposition](https://dangeng.github.io/factorized_diffusion/). *ECCV*, 2024.
- DeepFloyd. [IF](https://github.com/deep-floyd/IF): a pixel-space cascaded text-to-image diffusion model, 2023.
- H. A. Akitaya, E. D. Demaine, J. S. Ku. [Computing Flat-Folded States](https://erikdemaine.org/papers/FlatFolder_OSME2024/). *OSME*, 2024.